In [12]:
import os
import sys
from pathlib import Path

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

torch.manual_seed(8008135)

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("CODE_DIR:", CODE_DIR)
print("CODE_DIR contents:", os.listdir(CODE_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device set to {device}")

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

CODE_DIR: /home/daniel/HRM_Reconstruction/code
CODE_DIR contents: ['GPT2_Model', 'Utils', 'HRM_Model', 'Datasets', '.DS_Store', 'BiLSTM_Model', 'Sudoku', 'BERT_Model']
Device set to cuda


In [13]:
from Sudoku.sudoku import print_sudoku_comparison

from Datasets.Sudoku_DataLoader import get_loaders

from GPT2_Model.GPT2_Sudoku import GPT2_Baseline, GPT2Config
from GPT2_Model.GPT2_Train import train_gpt2

from Utils.schedules import cosine_schedule_with_warmup_lr_lambda
from Utils.checkpointing import load_checkpoint
from Utils.visualization import show_sudoku_predictions

In [14]:
train_size = 2**13
test_size = 2**8
batch_size = 2**8

train_loader, val_loader = get_loaders(
    train_size=train_size,
    test_size=test_size,
    batch_size=batch_size,
)

Map: 100%|██████████| 256/256 [00:00<00:00, 28157.07 examples/s]


In [15]:
model_config = GPT2Config(
    num_layers=12,
    num_heads=8,
    embedding_dim=512,
    vocab_size=10,
    block_size=81,
    dropout=0.2
)

lr = 3e-4
beta1 = 0.9
beta2 = 0.95
weight_decay = 0.1
num_epochs = 10

checkpoint_dir = "checkpoints"

In [16]:
model = GPT2_Baseline(model_config).to(device)

print(
    "Number of trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

Number of trainable parameters: 37,881,344


In [17]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

num_training_steps = len(train_loader) * num_epochs
num_warmup_steps = int(0.05 * num_training_steps)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_schedule_with_warmup_lr_lambda(
        step,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        min_ratio=0.1,
    ),
)

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)

num_training_steps: 320
num_warmup_steps: 16


In [18]:
model, best_metric, history = train_gpt2(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=5,
    validate_every=5,
    val_loader=val_loader,
    step_val_batches=1,
)

model.eval()

print("Best board accuracy used for checkpointing:", best_metric)

Number of trainable parameters: 37,881,344


Epoch 1: 100%|██████████| 32/32 [00:06<00:00,  4.62it/s]


Epoch 1: Avg Train Loss = 2.2297, Train Token Accuracy = 11.25%, Train Board Accuracy = 0.00%, LR = 2.98e-04


Epoch 2: 100%|██████████| 32/32 [00:06<00:00,  4.75it/s]


Epoch 2: Avg Train Loss = 2.1844, Train Token Accuracy = 12.59%, Train Board Accuracy = 0.00%, LR = 2.84e-04


...

Epoch 5: 100%|██████████| 32/32 [00:06<00:00,  4.74it/s]


Epoch 5: Avg Train Loss = 1.9862, Train Token Accuracy = 17.16%, Train Board Accuracy = 0.00%, LR = 1.76e-04


Validation: 100%|██████████| 1/1 [00:00<00:00, 13.96it/s]


Val Loss = 1.9137, Val Token Accuracy = 19.99%, Val Board Accuracy = 0.00%



KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history["step"], history["train_loss"], label="Train loss")

if any(v is not None for v in history["val_loss"]):
    plt.plot(history["step"], history["val_loss"], label="Small validation loss")

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Loss vs Training Step")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
show_sudoku_predictions(
    model,
    val_loader,
    device,
    print_sudoku_comparison,
    num_examples=10,
)

In [ ]:
torch.save(
    model.state_dict(),
    "checkpoints/gpt2_final_state_dict.pt",
)